# wav2vec2 training for voice-based order taking

Fine-tune `facebook/wav2vec2-base-960h` on SLURP + MInDS-14 for ASR, ready to be used inside a voice-based order-taking application.

In [1]:
import os

# Enable MPS fallback to CPU for CTC loss (not supported on MPS) - must be set BEFORE importing torch
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import random

import numpy as np
import torch

from datasets import load_dataset, Audio, concatenate_datasets
from transformers import (
    AutoProcessor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    set_seed,
)

In [2]:
SEED = 42
set_seed(SEED)

# Use CPU for training (CTC loss not supported on MPS/Metal)
# If you have NVIDIA GPU, PyTorch will automatically use CUDA
device = torch.device("cpu")

device


device(type='cpu')

In [3]:
MODEL_NAME = "facebook/wav2vec2-base-960h"

TARGET_SAMPLING_RATE = 16000

MINDS_LANG = "en-AU"

OUTPUT_DIR = "wav2vec2_order_taking"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
minds = load_dataset("PolyAI/minds14", name=MINDS_LANG)
minds

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 654
    })
})

In [5]:
slurp = load_dataset("qmeeus/slurp")
slurp

DatasetDict({
    train: Dataset({
        features: ['slurp_id', 'sentence', 'annotation', 'intent', 'audio'],
        num_rows: 50628
    })
    test: Dataset({
        features: ['slurp_id', 'sentence', 'annotation', 'intent', 'audio'],
        num_rows: 13078
    })
    devel: Dataset({
        features: ['slurp_id', 'sentence', 'annotation', 'intent', 'audio'],
        num_rows: 8690
    })
    train_synthetic: Dataset({
        features: ['slurp_id', 'sentence', 'annotation', 'intent', 'audio'],
        num_rows: 69253
    })
})

In [6]:
def add_transcript_column_minds(example):
    example["transcript"] = example["transcription"]
    return example


def add_transcript_column_slurp(example):
    example["transcript"] = example["sentence"]
    return example


minds = minds.map(add_transcript_column_minds)
slurp = {
    split_name: ds.map(add_transcript_column_slurp)
    for split_name, ds in slurp.items()
}

minds["train"][0], slurp["train"][0]

({'path': 'en-AU~PAY_BILL/response_4.wav',
  'audio': {'path': 'response_4.wav',
   'array': array([ 0.        ,  0.00024414, -0.00024414, ..., -0.00024414,
           0.00024414,  0.0012207 ]),
   'sampling_rate': 8000},
  'transcription': 'I would like to pay my electricity bill using my card can you please assist',
  'english_transcription': 'I would like to pay my electricity bill using my card can you please assist',
  'intent_class': 13,
  'lang_id': 2,
  'transcript': 'I would like to pay my electricity bill using my card can you please assist'},
 {'slurp_id': 9024,
  'sentence': 'event',
  'annotation': 'event',
  'intent': 10,
  'audio': {'path': 'audio-1501754435.flac',
   'array': array([ 0.00000000e+00, -3.05175781e-05,  0.00000000e+00, ...,
          -2.41149902e-01, -2.88116455e-01,  3.05175781e-05]),
   'sampling_rate': 16000},
  'transcript': 'event'})

In [7]:
minds = minds.cast_column("audio", Audio(sampling_rate=TARGET_SAMPLING_RATE))

slurp = {
    split_name: ds.cast_column("audio", Audio(sampling_rate=TARGET_SAMPLING_RATE))
    for split_name, ds in slurp.items()
}

minds["train"][0]["audio"]["sampling_rate"], slurp["train"][0]["audio"]["sampling_rate"]

(16000, 16000)

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

model.to(device)

processor, model.__class__.__name__

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


(Wav2Vec2Processor:
 - feature_extractor: Wav2Vec2FeatureExtractor {
   "do_normalize": true,
   "feature_extractor_type": "Wav2Vec2FeatureExtractor",
   "feature_size": 1,
   "padding_side": "right",
   "padding_value": 0.0,
   "return_attention_mask": false,
   "sampling_rate": 16000
 }
 
 - tokenizer: Wav2Vec2CTCTokenizer(name_or_path='facebook/wav2vec2-base-960h', vocab_size=32, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
 	0: AddedToken("<pad>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
 	1: AddedToken("<s>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
 	2: AddedToken("</s>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
 	3: AddedToken("<unk>", rstrip=

In [9]:
import re
import string


def normalize_text(text):
    text = text.lower()
    text = text.strip()

    allowed_chars = string.ascii_lowercase + string.digits + " '.,?!"
    text = "".join(ch for ch in text if ch in allowed_chars)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
def prepare_batch(batch):
    audio = batch["audio"]
    transcript = batch["transcript"]

    transcript = normalize_text(transcript)

    samples = audio["array"]
    sampling_rate = audio["sampling_rate"]

    inputs = processor(
        samples,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=False,
    )

    with processor.as_target_processor():
        labels = processor.tokenizer(
            transcript,
            return_tensors="pt",
        )

    batch["input_values"] = inputs.input_values[0]
    batch["labels"] = labels.input_ids[0]
    batch["input_length"] = len(batch["input_values"])

    return batch


In [11]:
NUM_PROC = os.cpu_count() or 1

# Check available splits
print("MINDS splits:", minds.keys())
print("SLURP splits:", slurp.keys())

# MINDS only has 'train', so split it into train/val
minds_train_split = minds["train"].train_test_split(test_size=0.2, seed=SEED)
minds_train = minds_train_split["train"].map(
    prepare_batch,
    remove_columns=minds_train_split["train"].column_names,
    num_proc=NUM_PROC,
)
minds_val = minds_train_split["test"].map(
    prepare_batch,
    remove_columns=minds_train_split["test"].column_names,
    num_proc=NUM_PROC,
)

# Prepare SLURP dataset
slurp_train = slurp["train"].map(
    prepare_batch,
    remove_columns=slurp["train"].column_names,
    num_proc=NUM_PROC,
)

# Use 'devel' for SLURP (development/validation set), or 'test' as fallback
slurp_eval_split = "devel" if "devel" in slurp else "test"
slurp_val = slurp[slurp_eval_split].map(
    prepare_batch,
    remove_columns=slurp[slurp_eval_split].column_names,
    num_proc=NUM_PROC,
)

train_dataset = concatenate_datasets([minds_train, slurp_train])
eval_dataset = concatenate_datasets([minds_val, slurp_val])

train_dataset[0], eval_dataset[0]


MINDS splits: dict_keys(['train'])
SLURP splits: dict_keys(['train', 'test', 'devel', 'train_synthetic'])


({'input_values': [0.00010747954365797341,
   -0.00463742483407259,
   -0.012932264246046543,
   -0.013419917784631252,
   -9.736445645103231e-05,
   0.014430709183216095,
   0.012583441101014614,
   -0.0032878974452614784,
   -0.01298872847110033,
   -0.007224843371659517,
   0.00020301411859691143,
   -0.004430113360285759,
   -0.013265205547213554,
   -0.010828359983861446,
   0.00048790217260830104,
   0.004962907172739506,
   -0.0009095901041291654,
   -0.004457884002476931,
   0.0007750466465950012,
   0.004799126647412777,
   -0.0011899960227310658,
   -0.010323400609195232,
   -0.011600923724472523,
   -0.005949507001787424,
   -0.0014404921093955636,
   -1.3112242413626518e-05,
   0.0012706260895356536,
   0.0014405455440282822,
   -0.0016395313432440162,
   -0.003538151504471898,
   0.0014364277012646198,
   0.009326654486358166,
   0.010876553133130074,
   0.005613813642412424,
   0.00152285723015666,
   0.001013196655549109,
   -0.0018090594094246626,
   -0.0083121778443455

In [12]:
MAX_SECONDS = 15
MAX_LENGTH = MAX_SECONDS * TARGET_SAMPLING_RATE


def filter_long_examples(batch):
    return batch["input_length"] < MAX_LENGTH


train_dataset = train_dataset.filter(filter_long_examples)
eval_dataset = eval_dataset.filter(filter_long_examples)

len(train_dataset), len(eval_dataset)

(51079, 8806)

In [13]:
import torch
from dataclasses import dataclass
from typing import Dict, List, Optional, Union


@dataclass
class DataCollatorCTCWithPadding:
    """Custom data collator for CTC with padding."""
    processor: object
    padding: Union[bool, str] = True
    max_length: Optional[int] = None
    max_length_labels: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    pad_to_multiple_of_labels: Optional[int] = None

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate input_values and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad input_values
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                max_length=self.max_length_labels,
                pad_to_multiple_of=self.pad_to_multiple_of_labels,
                return_tensors="pt",
            )

        # Replace padding with -100 to ignore in loss computation
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(processor=processor)


In [14]:
def _levenshtein_distance(ref_tokens, hyp_tokens):
    m = len(ref_tokens)
    n = len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_tokens[i - 1] == hyp_tokens[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[m][n]


def _word_error_rate_from_strings(ref, hyp):
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    if len(ref_tokens) == 0:
        return 0.0 if len(hyp_tokens) == 0 else 1.0
    distance = _levenshtein_distance(ref_tokens, hyp_tokens)
    return float(distance) / float(len(ref_tokens))


def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wers = [
        _word_error_rate_from_strings(r, p)
        for r, p in zip(label_str, pred_str)
    ]
    wer = float(sum(wers)) / float(len(wers)) if len(wers) > 0 else 0.0

    return {"wer": wer}

In [15]:
BATCH_SIZE = 2
EVAL_STEPS = 500
SAVE_STEPS = 500
LOG_STEPS = 100
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    group_by_length=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    save_strategy="steps",
    num_train_epochs=NUM_EPOCHS,
    fp16=torch.cuda.is_available(),
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=LOG_STEPS,
    save_total_limit=3,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)
training_args


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer

/var/folders/jz/24tmydbj57d72q4g5hl2pqyr0000gn/T/ipykernel_3981/1063297457.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
train_result = trainer.train()
train_result

KeyboardInterrupt: 

In [18]:
metrics = trainer.evaluate()
metrics

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/lib/python3.13/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torch/nn/functional.py:3071: UserWarning: The operator 'aten::_ctc_loss' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:15.)
  return torch.ctc_loss(


{'eval_loss': inf,
 'eval_model_preparation_time': 0.0011,
 'eval_wer': 1.0638301152665885,
 'eval_runtime': 703.6929,
 'eval_samples_per_second': 12.514,
 'eval_steps_per_second': 6.257}

In [19]:
best_model_dir = os.path.join(OUTPUT_DIR, "best")
os.makedirs(best_model_dir, exist_ok=True)

trainer.save_model(best_model_dir)
processor.save_pretrained(best_model_dir)

best_model_dir

'wav2vec2_order_taking/best'

In [20]:
from datasets import Audio as LocalAudio
import soundfile as sf


inference_processor = AutoProcessor.from_pretrained(best_model_dir)
inference_model = Wav2Vec2ForCTC.from_pretrained(best_model_dir).to(device)
inference_model.eval()


def transcribe_audio_file(path_to_audio: str) -> str:
    samples, sampling_rate = sf.read(path_to_audio)

    inputs = inference_processor(
        samples,
        sampling_rate=sampling_rate,
        return_tensors="pt",
        padding=True,
    )

    with torch.no_grad():
        logits = inference_model(
            inputs.input_values.to(device)
        ).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = inference_processor.batch_decode(predicted_ids)[0]

    transcription = normalize_text(transcription)

    return transcription